# Lesson 1 — Your first model

**Goal:** go through the whole machine-learning loop once (data → features → train → evaluate → look inside) on your own transactions.

Supervised learning in one sentence: show a model many *(input, correct answer)* pairs, and it learns a function that maps new inputs to answers. Here the input is an SMS and the answer is a category.

**The catch:** you have no correct answers yet; the test set isn't labelled. But every transaction has the keyword rules' guess, so in this lesson the rules are the model's teacher. That's enough to learn every mechanic, and the result teaches the most important lesson in ML (section 7).

Run a cell with **Shift+Enter**. Where you see **Predict**, write your guess down *before* running. That's where the learning happens.

In [ ]:
import json
from pathlib import Path

import pandas as pd

DATA = Path("../data")
df = pd.read_json(DATA / "dataset.jsonl", lines=True)
test_ids = set(json.loads((DATA / "test_ids.json").read_text()))
df["is_test"] = df["id"].isin(test_ids)
pd.set_option("display.max_colwidth", 120)
df.shape  # (rows, columns)

## 1. Look at the data first

Every ML project starts here. Models fail in ways that only make sense once you know what the data looks like.

**Predict:** which category do the rules assign most often, and roughly what share?

In [ ]:
df["predicted"].value_counts()

In [ ]:
df.sample(8, random_state=0)[["date", "sender", "direction", "amountMinor", "merchant", "predicted"]]

**Try:** change `random_state` a few times, and read some bodies with `df.sample(3)["body"].tolist()`. Which words would *you* use to decide the category?

Amount is a clue too: airtime is small, rent is big. This lesson ignores it; it comes back later.

In [ ]:
(df["amountMinor"] / 100).groupby(df["predicted"]).describe()[["count", "50%", "max"]].round(0)

## 2. Features: turning text into numbers

A model can only do arithmetic, so each SMS has to become a list of numbers: a **feature vector**.

The simplest idea is counting words. We use **character n-grams** instead, every run of 3–5 characters: "Supermarket" becomes `sup`, `upe`, `per`, … They work for Amharic without a tokenizer and survive typos and abbreviations ("SuperMkt").

**TF-IDF** then weights each n-gram: frequent in this message (TF) but rare across all messages (IDF) means important. "Dear" is in almost every SMS, so it counts for little.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

demo = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 3))
X_demo = demo.fit_transform(["paid to Shoa Supermarket", "transferred to Abebe"])
print(X_demo.shape)  # (messages, features)
print(demo.get_feature_names_out()[:15])

**Predict:** what happens to the number of features with `ngram_range=(2, 4)`? Try it, then put it back.

## 3. Split: train, validation, test

Grade a model on the examples it learned from and you're testing memory, not understanding. So the data is split:

| set | used for | here |
|---|---|---|
| train | learning the weights | 80% of the non-test rows |
| validation | comparing experiments | 20% of the non-test rows |
| test | the final exam, scored rarely | the 300 frozen `test_ids` you'll label |

The test set is not touched in this lesson.

In [ ]:
from sklearn.model_selection import train_test_split

df["text"] = df["direction"] + " " + df["body"]
pool = df[~df["is_test"]]
train, val = train_test_split(pool, test_size=0.2, random_state=42)
len(train), len(val)

## 4. Train

**Logistic regression** is the workhorse classifier. For each category it learns one weight per n-gram. To classify a message, add up the weights of the n-grams it contains, once per category; the highest total wins (a softmax turns the totals into probabilities).

So the trained model is a big table of numbers plus an addition. In step 2 you'll run exactly this table in Dart, on the phone.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

model = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2),
    LogisticRegression(max_iter=2000),
)
model.fit(train["text"], train["predicted"])  # the teacher: the rules' guesses
vectorizer, classifier = model.named_steps.values()
print(len(vectorizer.get_feature_names_out()), "features ×", len(classifier.classes_), "categories")

## 5. Evaluate on the validation set

**Predict:** on messages it has never seen, how often will the model agree with the rules? 50%? 80%? 99%?

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

val_pred = model.predict(val["text"])
print(f"agreement with the rules: {accuracy_score(val['predicted'], val_pred):.1%}")
print(classification_report(val["predicted"], val_pred, zero_division=0))

- **Precision:** when the model says *food*, how often is it food?
- **Recall:** of all the *food*, how much did it find?
- **Support:** how many examples there were. With fewer than ~5, the scores are noise.

Rare categories usually score worst: the model barely saw them.

**Try:** `min_df=1` or `ngram_range=(2, 4)`, retrain, and compare. That's what the validation set is for.

## 6. Look inside the model

Nobody gave this model any keywords. Here are the n-grams that push hardest toward each category:

In [ ]:
names = vectorizer.get_feature_names_out()
for i, category in enumerate(classifier.classes_):
    top = classifier.coef_[i].argsort()[-8:][::-1]
    print(f"{category:<14}", " | ".join(repr(names[j]) for j in top))

Compare with the keyword lists in `inferCategory` (`lib/features/sms_ingestion/domain/services/sms_parser.dart`). It rediscovered many of them from examples alone; that's what "learning" means here.

You'll also see junk: bank names, boilerplate. The model uses *any* pattern that goes with the answer, sensible or not. That's a **spurious correlation**, and the reason you always look inside.

## 7. Error analysis

The most valuable skill in applied ML: read the mistakes one by one.

In [ ]:
val = val.assign(model_says=val_pred)
disagree = val[val["predicted"] != val["model_says"]]
print(len(disagree), "disagreements")
disagree[["predicted", "model_says", "body"]].head(15)

**Try:** for each row, decide who's right, the rules or the model, and keep a tally. Sometimes it's the model.

Now the transactions where the rules gave up and only knew the direction:

In [ ]:
gave_up = val[val["predicted"].isin(["expense", "income"])]
probs = model.predict_proba(gave_up["text"])
pd.DataFrame({
    "model_says": classifier.classes_[probs.argmax(axis=1)],
    "confidence": probs.max(axis=1).round(2),
    "body": gave_up["body"].str[:90],
}).head(10)

Mostly `expense` and `income` again. Of course: its teacher answers "expense" for these, so it does too.

**The lesson:** a model is only as good as its labels. Trained on the rules, it imitates the rules, mistakes included. High agreement with them says nothing about whether either is *right*. To beat the rules, the model needs answers the rules don't have: **yours**.

## Next

1. Label the test set (`python3 tool/ml/label.py`), so "right" can be measured.
2. Lesson 2: collect training labels fast (label a counterparty once and it covers all its transactions), retrain on *your* labels, and score on the test set against `baseline.json`.